Import Libraries

In [9]:
import torch
import torch.nn as nn

import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F

In [10]:
class ModelArchitecture(nn.Module):
    def __init__(
        self, 
        num_channels: int = 3, 
        num_classes: int = 3
    ):
        super(ModelArchitecture, self).__init__()

        # Conv1
        self.conv1 = nn.Conv2d(num_channels, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.shortcut1 = nn.Sequential(
            nn.Conv2d(num_channels, 32, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(32)
        )
        
        # Conv2
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64) 
        self.shortcut2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(64)
        )
        # Conv3
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128) 
        self.shortcut3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(128)
        )
        
        self.pool = nn.MaxPool2d(2, 2) 

        # Fully Connected Network
        self.fc1 = nn.Linear(128*5*17, 32) 
        self.fc2 = nn.Linear(32, num_classes)

        self.dropout = nn.Dropout(p=0.3)  
        self.dropout2d = nn.Dropout2d(p=0.5)

    def forward(self, x):
        res = self.shortcut1(x)
        out = self.bn1(self.conv1(x))
        out += res
        x = self.dropout2d(self.pool(F.relu(out)))   

        res = self.shortcut2(x)
        out = self.bn2(self.conv2(x))
        out += res
        x = self.dropout2d(self.pool(F.relu(out)))

        
        res = self.shortcut3(x)
        out = self.bn3(self.conv3(x))
        out += res
        x = self.dropout2d(self.pool(F.relu(out)))

        x = x.view(x.size(0), -1) 

        # Feed forward network
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [ ]:
# Define Model
model_path = r'model.pth'
model_dict = torch.load(model_path, map_location='cpu')

model = ModelArchitecture(3, 3) # three channels and 3 output classes
model.load_state_dict(model_dict)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4576\1136610083.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_dict = torch.load(model_path, map_location='cpu')


<All keys matched successfully>

In [12]:
def pad_waveform(
    waveform: torch.tensor, 
    target_length: (int) = int(4.5 * 16000)
):
    """
    Pad the waveform to window_size.

    Arg(s):
        waveform (Tensor):  Waveform tensor of size [1, n]
        target_length (int): Target length for pad
    Return:
        padded_waveform (Tensor): Padded waveform of size [1, target_length]
    """
    current_length = waveform.shape[1]

    if current_length < target_length:
        padding = target_length - current_length

        waveform = torch.nn.functional.pad(
            waveform,
            (0, padding)
        )

    else:
        waveform = waveform[:, :target_length]

    return waveform

In [13]:
mfcc_transform = T.MFCC(
    sample_rate=16000,
    n_mfcc=40,
    melkwargs={
        "n_fft": 1024,
        "hop_length": 512,
        "n_mels": 128,
    }
)

def transform(
    waveform: torch.tensor = None,
    sr: int = None
):
    resampler = T.Resample(
        orig_freq=sr,
        new_freq=16000
    )
    waveform = resampler(waveform)
    waveform = pad_waveform(waveform)

    mfcc = mfcc_transform(waveform)
    delta = torchaudio.functional.compute_deltas(mfcc)
    delta2 = torchaudio.functional.compute_deltas(delta)

    mfcc_features = torch.cat(
        [mfcc, delta, delta2],
        dim=0
    )

    return mfcc_features.unsqueeze(0) # Our model excepts 4D input of shape (batch_size, channels, height, weight)

In [14]:
def inference(
    tensor: torch.tensor
):
    model.eval()
    with torch.no_grad():
        output = model(tensor)

    return torch.argmax(output, dim=1).item()

In [15]:
def predict(
    audio_path: str = None
):
    waveform, sr = torchaudio.load(audio_path) # Load audio file

    transformed_audio = transform(waveform, sr)

    pred = inference(transformed_audio) # Make inference
    class_ = 'COPD' if pred == 0 else 'Healthy' if pred == 1 else 'Pneumonia'

    return f"Predicted class: {class_}"


In [16]:
audio_path = r"ICBHI_final_database\201_1b1_Al_sc_Meditron.wav"
predict(audio_path)

'Predicted class: COPD'